---
## Hausaufgabe 3
---

### Achtung: Überprüfen Sie vor Abgabe der Hausaufgabe, ob das Notebook richtig gespeichert wurde. 
Die Speicherung von Notebooks funktioniert über das "Disketten"-Symbol im Notebook oder über die Shortcuts Strg + S (Win) und CMD + S (MAC).

---
## Bewertung
---

#### Erreichbare Punkte: 25
---
#### Erreichte Punkte:   -
---
---
Aufgabe 1:<br>
<br>
Aufgabe 2:<br>

---

#### Kontext - Das Job Shop Problem

Neben dem Flow Shop Problem ist das Job Shop Problem eines der am weitesten verbreiteten Problemstellungen im Bereich der Maschinenbelegungsplanung. Im Unterschied zum Flow Shop existieren beim Job Shop unterschiedliche Bearbeitungsreihenfolgen für jeden Auftrag. Bspw. könnte Auftrag 1 die Bearbeitungsreihenfolge M1 - M3 - M2 haben, während Auftrag 2 Reihenfolge M3 - M1 - M2 benötigt.

Ausgehend von dieser Beschreibung beschäftigt sich die nachfolgende Hausaufgabe explizit mit dem **JOB SHOP PPROBLEM**.

---

#### Aufgabe 1 - Datenimport und Bewertung (12 Punkte)

##### a.) (2 Punkte)

Machen Sie sich mit den Dateien InputData.py, OutputData.py, JSP.json, und read.me vertraut. Importieren Sie anschließend alle Klassen und Objekte aus den mit gelieferten .py-Dateien. Erzeugen Sie anschließend das Objekt **data** als Instanz der Klasse *InputData*. Als Inputdaten dienen Ihnen die Daten aus der mitgelieferten .json-Datei (JSP.json). 


Lassen Sie sich anschließend jeden InputJob mit den Operationen, Zeiten und den zugehörigen Maschinen anzeigen.


In [ ]:
import json
from InputData import *
from OutputData import *

data = InputData('../../../data/Prescriptive_Homework/ha_3/JSP.json')
# short version to print all jobs in data.InputJobs
# print('\n'.join(str(job) for job in data.InputJobs))

# long version to print all jobs in data.InputJobs
for job in data.InputJobs:
    print(job)

Job 1 with 3 Operations:
Operation 0 on machine 0 with Processingtime: 4 
Operation 1 on machine 1 with Processingtime: 9 
Operation 2 on machine 2 with Processingtime: 6 

Job 2 with 3 Operations:
Operation 0 on machine 1 with Processingtime: 5 
Operation 1 on machine 2 with Processingtime: 1 
Operation 2 on machine 0 with Processingtime: 11 

Job 3 with 3 Operations:
Operation 0 on machine 1 with Processingtime: 6 
Operation 1 on machine 0 with Processingtime: 14 
Operation 2 on machine 2 with Processingtime: 7 



##### b.) (2 Punkte)

Objekte der Klasse *Solution* benötigen bei Intialisierung einen Parameter *permutation*. Dieser stellt eine Sequenz mit Wiederholung dar: Eine JobId wird in dieser Sequenz entsprechend Ihrer Operations-/ Maschinenanzahl wiederholt. Für die Beispieldatei *JSP.json* ergibt sich somit eine Sequenz aus neun Ziffern (3x3).

**Beispiel**

<details>
    - Es sind drei Aufträge (Jobs) und zwei Maschinen gegeben.<br>
    - Eine gültige Permutation ergibt sich somit aus 6 Ziffern: bspw. [1, 2, 3, 1, 2, 3]<br>
    - Die erste "1" stellt in der Permutation die erste Operation des Auftrags 1 dar, die zweite "1" die zweite Operation, usw.<br>
    - In dem genannten Beispiel würde als erstes die erste Operation von Auftrag 1, dann die erste Operation von Auftrag 2 eingeplant werden.
</details>

Erstellen Sie ein Objekt **simpleSolution** der Klasse *Solution*, bei dem die Aufträge für alle Maschinen auf jeden Fall die Reihenfolge 1 - 2 - 3 gewählt wird.

In [9]:
simpleSolution = Solution(data.InputJobs, [1,1,1,2,2,2,3,3,3])
print(simpleSolution)

The permutation [1, 1, 1, 2, 2, 2, 3, 3, 3] results in a Makespan of -1


##### c.) (8 Punkte)

Die Bewertungslogik für die Ermittlung der Makespan bei Flow Shop Problemen kennen Sie bereits aus dem Seminar. Der dazugehörige Code ist unten nochmals aufgeführt. Formulieren Sie diesen Code für den Fall des Job Shop Problems um und bestimmen Sie die Makespan von **simpleSolution**.

In [ ]:
import numpy 

class EvaluationLogic:    
    def DefineStartEnd(self, currentSolution):    
        #####
        # schedule first job: starts when finished at previous stage
        firstJob = currentSolution.OutputJobs[currentSolution.Permutation[0]]
        firstJob.EndTimes = numpy.cumsum([firstJob.ProcessingTime(x) for x in range(len(firstJob.EndTimes))])
        firstJob.StartTimes[1:] = firstJob.EndTimes[:-1]
        #####
        # schedule further jobs: starts when finished at previous stage and the predecessor is no longer on the considered machine
        for j in range(1,len(currentSolution.Permutation)):
            currentJob = currentSolution.OutputJobs[currentSolution.Permutation[j]]
            previousJob = currentSolution.OutputJobs[currentSolution.Permutation[j-1]]
            # first machine
            currentJob.StartTimes[0] = previousJob.EndTimes[0]
            currentJob.EndTimes[0] = currentJob.StartTimes[0] + currentJob.ProcessingTime(0)
            # other machines
            for i in range(1,len(currentJob.StartTimes)):
                currentJob.StartTimes[i] = max(previousJob.EndTimes[i], currentJob.EndTimes[i-1])
                currentJob.EndTimes[i] = currentJob.StartTimes[i] + currentJob.ProcessingTime(i)
        #####
        # Save Makespan and return Solution
        currentSolution.Makespan = currentSolution.OutputJobs[currentSolution.Permutation[-1]].EndTimes[-1]

##### Lösung mit Copilot

In [ ]:
# Start and End Times + Makespan of Job SHop Problem
class EvaluationLogic:
    def DefineStartEnd(self, currentSolution):
        # Track last end time for each machine
        machine_last_end = {}
        # Track last end time for each job
        job_last_end = {}
        # Track how many times each job has appeared so far (operation index)
        job_operation_count = {job_id: 0 for job_id in currentSolution.OutputJobs}

        # Iterate through the permutation
        for job_id in currentSolution.Permutation:
            job = currentSolution.OutputJobs[job_id-1]
            op_idx = job_operation_count[job_id-1]  # which operation of this job

            machine_id = job.MachineSequence[op_idx]
            processing_time = job.ProcessingTimes[machine_id]

            # Get last end time for this machine and this job
            last_machine_end = machine_last_end.get(machine_id, 0)
            last_job_end = job_last_end.get(job_id-1, 0)

            # Start time: max of when machine is free and when job is ready
            start_time = max(last_machine_end, last_job_end)
            end_time = start_time + processing_time

            # Save start and end times for this operation
            job.StartTimes[op_idx] = start_time
            job.EndTimes[op_idx] = end_time

            # Update trackers
            machine_last_end[machine_id] = end_time
            job_last_end[job_id-1] = end_time
            job_operation_count[job_id-1] += 1

        # Makespan: max end time of all jobs' last operation
        currentSolution.Makespan = max(
            job.EndTimes[-1] for job in currentSolution.OutputJobs.values()
        )

In [18]:
EvaluationLogic().DefineStartEnd(simpleSolution)
print(simpleSolution)

The permutation [1, 1, 1, 2, 2, 2, 3, 3, 3] results in a Makespan of 52


##### Eigene Lösung
**Pseudo Code** ausgehend von Permutationsreihenfolge:
1. Welcher Job wird gerade betrachtet
2. Welche Operation dieses Jobs wird ausgeführt
3. Welche Maschine wird für diese Operation benötigt
4. Was ist die Bearbeitungszeit dieser Operation auf dieser Maschine vom betrachteten Job
5. Wann sind die Maschine (vorheriger Job) und der Job (vorherige Maschine) verfügbar
6. Bestimme Start- (max aus Maschinen- und Jobverfügbarkeit) und Endzeit

In [24]:
class EvalutaionsLogik:
    def DefineStartEnd(self, currentSolution):
        # count appearance (i.e. operations) of each job in permutation
        count_joboperations = [0 for _ in currentSolution.OutputJobs]
        # track machine time
        machine_times = {}
        # track job time
        job_times = {}

        for jobid in currentSolution.Permutation:
            # 1) which job is it
            job = currentSolution.OutputJobs[jobid-1]
            # 2) which operation of this job
            operation_idx = count_joboperations[jobid-1]
            # 3) which machine is needed
            machine_id = job.MachineSequence[operation_idx]
            # 4) processing time on this machine of this operation of this job
            processing_time = job.ProcessingTimes[machine_id]

            # 5) When is machine (= finished previous job) and job ready (= finished at previous stage)
            machine_ready = machine_times.get(machine_id, 0)    # default 0 if machine not used yet
            job_ready = job_times.get(jobid-1, 0)   # default 0 if job not processed yet

            # 6) Determine start and end time of this operation of this job and save it to OutputJobs
            start_time = max(machine_ready, job_ready)
            end_time = start_time + processing_time
            job.StartTimes[operation_idx] = start_time
            job.EndTimes[operation_idx] = end_time

            # 7) update and save new machine and job times
            machine_times[machine_id] = end_time
            job_times[jobid-1] = end_time

            # 8) update operation count of the current job
            count_joboperations[jobid-1] += 1

        # 9) Determine and save makespan of the current solution
        # currentSolution.Makespan = max(list(machine_times.values()) + list(job_times.values()))
        currentSolution.Makespan = max(currentSolution.OutputJobs[jobid].EndTimes[-1] for jobid in currentSolution.OutputJobs.keys())

In [35]:
EvalutaionsLogik().DefineStartEnd(simpleSolution)
print(simpleSolution)

The permutation [1, 1, 1, 2, 2, 2, 3, 3, 3] results in a Makespan of 52


Erwarteter Output:

    The permutation [1, 1, 1, 2, 2, 2, 3, 3, 3] results in a Makespan of 52

In [27]:
simpleSolution.OutputJobs.values()

dict_values([<OutputData.OutputJob object at 0x000001CBC2CE6F10>, <OutputData.OutputJob object at 0x000001CBC2CE6CF0>, <OutputData.OutputJob object at 0x000001CBC2C8B550>])

In [34]:
print('Startzeiten der Jobs:')
for job in simpleSolution.OutputJobs.values():
    print(str('Job'), job.JobId, str(':'), job.StartTimes)

print('Endzeiten der Jobs:')
for job in simpleSolution.OutputJobs.values():
    print(str('Job'), job.JobId, str(':'), job.EndTimes)

print('Maschinenreihenfolge der Jobs:')
for job in simpleSolution.OutputJobs.values():
    print(str('Job'), job.JobId, str(':'), job.MachineSequence)

print('Bearbeitungszeiten der Jobs:')
for job in simpleSolution.OutputJobs.values():
    print(str('Job'), job.JobId, str(':'), job.ProcessingTimes)

Startzeiten der Jobs:
Job 1 : [0, 4, 13]
Job 2 : [13, 19, 20]
Job 3 : [18, 31, 45]
Endzeiten der Jobs:
Job 1 : [4, 13, 19]
Job 2 : [18, 20, 31]
Job 3 : [24, 45, 52]
Maschinenreihenfolge der Jobs:
Job 1 : [0, 1, 2]
Job 2 : [1, 2, 0]
Job 3 : [1, 0, 2]
Bearbeitungszeiten der Jobs:
Job 1 : [4, 9, 6]
Job 2 : [11, 5, 1]
Job 3 : [14, 6, 7]


#### Aufgabe 2 - Das Shifting-Bottleneck-Verfahren (13 Punkte)
Das Shifting-Bottleneck-Verfahren zählt zu den besten Eröffnungsverfahren für das Job Shop Problem. Nach der erstmaligen Entwicklung durch Adams, Balas und Zawack im Jahr 1988, wurde das Verfahren auf eine Vielzahl von anderen Problemen angewendet und weiterentwickelt.

Die Grundidee besteht darin, dass in jeder Iteration eine bislang noch nicht eingeplante Maschine als Engpass identifiziert wird und anschließend die Reihenfolge der Aufträge auf dieser Maschine bestimmt wird.

---

##### a.) Vorbereitende Maßnahmen (4 Punkte)

Extrahieren Sie die Bearbeitungszeiten und die Maschinensequenzen für alle Aufträge aus **data** und speichern Sie diese Informationen in zwei separate Listen **processingTimes** und **machineSequences** als geschachtelte Listen oder Matrizen ab. Ermitteln Sie im Anschluss für jede Maschine einzeln die Vorlauf-, Bearbeitungs- und Nachlaufzeiten für jeden Auftrag:

**Vorlaufzeiten (V)**: Summe aller Bearbeitungszeiten, die aufgrund der Maschinensequenz vor der aktuellen Operation erfolgt sein müssen.<br>
**Bearbeitungszeiten (B)**: Bearbeitungszeit der aktuellen Operation.<br>
**Nachlaufzeiten (N)**: Summe aller Bearbeitungszeiten, die aufgrund der Maschinensequenz nach der aktuellen Operation erfolgen.<br>

Speichern Sie Ihre Ergebnisse als Dictionary und nutzen Sie den Maschinenindex als übergeordneten *key* und als *value* ein weiteres Dictionary mit den *keys* **V, B, N** für die Zeiten.

##### Pseudo Code Maschinenzeiten
1. Wähle Maschinenindex
2. Bestimme Bearbeitungszeit der Maschine für jeden Job
3. Bestimme Position des Maschinenindex in der Maschinensequenz
4. Bestimme Operationen vor und nach dem Maschinenindex in der Maschinensequenz
5. Berechne Vorlaufzeit und Nachlaufzeit auf Grundlage der Bearitungszeiten

##### Matrizen

In [92]:
import numpy as np

processingTimes = np.array([job.ProcessingTimes for job in data.InputJobs])
machineSequences = np.array([job.MachineSequence for job in data.InputJobs])
print('Bearbeitungszeiten der Jobs (Matrix):\n', processingTimes)
print('Maschinenreihenfolgen der Jobs (Matrix):\n', machineSequences)

Bearbeitungszeiten der Jobs (Matrix):
 [[ 4  9  6]
 [11  5  1]
 [14  6  7]]
Maschinenreihenfolgen der Jobs (Matrix):
 [[0 1 2]
 [1 2 0]
 [1 0 2]]


In [94]:
# Vorlauf, Nachlauf- und Bearbeitungszeiten der Maschinen
# ...existing code...
machineTimes = {}

# Assume processingTimes and machineSequences are np.arrays
for machine in range(data.m):
    vorlaufzeiten = []
    bearbeitungszeiten = []
    nachlaufzeiten = []
    for job in range(data.n):
        # machineSequences[job] is a np.array of machine indices for this job
        if machine in machineSequences[job]:
            operation_idx = np.where(machineSequences[job] == machine)[0][0]
            # Vorlaufzeit: sum of processing times before op_idx in the job's machine sequence
            vorlauf = np.sum([processingTimes[job][machineSequences[job][i]] for i in range(operation_idx)])
            # Bearbeitungszeit: processing time at op_idx
            bearbeitung = processingTimes[job][machineSequences[job][operation_idx]]
            # Nachlaufzeit: sum of processing times after op_idx in the job's machine sequence
            nachlauf = np.sum([processingTimes[job][machineSequences[job][i]] for i in range(operation_idx+1, len(machineSequences[job]))])
            vorlaufzeiten.append(vorlauf)
            bearbeitungszeiten.append(bearbeitung)
            nachlaufzeiten.append(nachlauf)
    machineTimes[machine] = {'V': vorlaufzeiten, 'B': bearbeitungszeiten, 'N': nachlaufzeiten}

print(machineTimes)

{0: {'V': [np.float64(0.0), np.int64(6), np.int64(6)], 'B': [np.int64(4), np.int64(11), np.int64(14)], 'N': [np.int64(15), np.float64(0.0), np.int64(7)]}, 1: {'V': [np.int64(4), np.float64(0.0), np.float64(0.0)], 'B': [np.int64(9), np.int64(5), np.int64(6)], 'N': [np.int64(6), np.int64(12), np.int64(21)]}, 2: {'V': [np.int64(13), np.int64(5), np.int64(20)], 'B': [np.int64(6), np.int64(1), np.int64(7)], 'N': [np.float64(0.0), np.int64(11), np.float64(0.0)]}}


##### Geschachtelte Listen 
-> eher ungewöhnlich und prinzipiell in OR nicht empfohlen. Gurobi erwartet bspw. ebenfalls np.arrays

In [85]:
processingTimes = [job.ProcessingTimes for job in data.InputJobs]
machineSequences = [job.MachineSequence for job in data.InputJobs]
print('Bearbeitungszeiten der Jobs:', processingTimes)
print('Maschinenreihenfolgen der Jobs:', machineSequences)

Bearbeitungszeiten der Jobs: [[4, 9, 6], [11, 5, 1], [14, 6, 7]]
Maschinenreihenfolgen der Jobs: [[0, 1, 2], [1, 2, 0], [1, 0, 2]]


In [91]:
# Vorlauf, Nachlauf- und Bearbeitungszeiten der Maschinen
machineTimes = {}

# iterate over all machine IDs
for machine in range(data.m):
    # define empty lists to store times
    vorlaufzeiten = []
    bearbeitungszeiten = []
    nachlaufzeiten = []
    # iterate over all jobs to find those that use the current machine
    for job in range(data.n):
        # check if the current machine is in the job's machine sequence
        if machine in machineSequences[job]:
            # find the index of the operation on this machine
            operation_idx = machineSequences[job].index(machine)
            # Vorlaufzeit: sum of processing times before op_idx in the jobs machine sequence
            vorlauf = sum([processingTimes[job][machineSequences[job][machineid]] for machineid in range(operation_idx)])
            bearbeitung = processingTimes[job][machineSequences[job][operation_idx]]
            # Nachlaufzeit: sum of processing times after op_idx in the jobs machine sequence
            nachlauf = sum([processingTimes[job][machineSequences[job][machineid]] for machineid in range(operation_idx+1, len(machineSequences[job]))])
            # append times to respective lists
            vorlaufzeiten.append(vorlauf)
            bearbeitungszeiten.append(bearbeitung)
            nachlaufzeiten.append(nachlauf)
    # store the lists in the dictionary for the current machine
    machineTimes[machine] = {'V': vorlaufzeiten, 'B': bearbeitungszeiten, 'N': nachlaufzeiten}

print(machineTimes)

{0: {'V': [0, 6, 6], 'B': [4, 11, 14], 'N': [15, 0, 7]}, 1: {'V': [4, 0, 0], 'B': [9, 5, 6], 'N': [6, 12, 21]}, 2: {'V': [13, 5, 20], 'B': [6, 1, 7], 'N': [0, 11, 0]}}


##### Best practice Musterlösung

In [95]:
processingTimes = []
machineSequences = []

for job in data.InputJobs:
    processingTimes.append(job.ProcessingTimes)
    machineSequences.append(job.MachineSequence)

# Initialisiere ein leeres Dictionary für die Ergebnisse
machineTimes = {}

# Iteriere über jede Maschine
for machineIndex in range(len(processingTimes[0])):
    # Initialisiere Vorlaufzeiten (V), Bearbeitungszeiten (B) und Nachlaufzeiten (N) für diese Maschine
    V = []
    B = []
    N = []

    # Iteriere über jeden Auftrag
    for jobIndex in range(len(processingTimes)):
        # Ermittle die Bearbeitungszeit der aktuellen Operation
        processing_time_current = processingTimes[jobIndex][machineIndex]
        B.append(processing_time_current)

        # Ermittle die Maschinensequenz für diesen Auftrag
        machine_sequence_current = machineSequences[jobIndex].index(machineIndex)

        # Ermittle die Vorlaufzeiten
        V_sum = 0
        for i in machineSequences[jobIndex][:machine_sequence_current]:
            V_sum += processingTimes[jobIndex][i]
        V.append(V_sum)

        # Ermittle die Nachlaufzeiten
        N_sum = 0
        for i in machineSequences[jobIndex][machine_sequence_current+1:]:
            N_sum += processingTimes[jobIndex][i]
        N.append(N_sum)


    # Speichere die Zeiten für diese Maschine im Ergebnis-Dictionary
    machineTimes[machineIndex] = {'V': V, 'B': B, 'N': N}

print(machineTimes)

{0: {'V': [0, 6, 6], 'B': [4, 11, 14], 'N': [15, 0, 7]}, 1: {'V': [4, 0, 0], 'B': [9, 5, 6], 'N': [6, 12, 21]}, 2: {'V': [13, 5, 20], 'B': [6, 1, 7], 'N': [0, 11, 0]}}


##### b.) Das Verfahren von Schrage (9 Punkte)

Das Verfahren von Schrage ist dient der heuristischen Minimierung der Zyklusdauer bei Vor- und Nachlaufzeiten im **Ein-Maschinen-Fall**. Das Ende der Zyklusdauer ist erreicht, wenn die Nachlaufzeit aller Aufträge abgelaufen ist. In unserem Beispiel wollen wir mit diesem Verfahren die Engpass-Maschine ermitteln.

Vorgehensweise des Verfahrens:<br>

1. Die Maschinenbelegung beginnt am Anfang des Planungszeitraums mit anschließender sukzessiver Auftragsauswahl.
2. Ein noch nicht eingeplanter Auftrag kann entweder einplanbar oder aufgrund seiner Vorlaufzeit noch nicht einplanbar sein.
3. Zu jedem Zeitpunkt, zu dem die Maschine frei ist, wird aus der Menge der gegenwärtig einplanbaren Aufträge stets der Auftrag eingeplant, dessen Nachlaufzeit am größten ist.

Ziel: Aufträge mit langer Nachlaufzeit sollen möglichst frühzeitig eingeplant werden.

Schreiben Sie eine Funktion **getBottleneck()**, die das Dictionary aus Aufgabe a als Argument erhält und anschließend für jede Maschine das Verfahren von Schrage ausführt. Im Anschluss soll die Funktion die aktuelle Engpass-Maschine zurückgeben. Testen Sie die entwickelte Funktion mit den Beispieldaten.

**Tipp:**

<details>
- Ein Beispiel des Verfahrens finden Sie in der OPM-Vorlesung Kapitel 6 (Beigefügter Ausschnitt) <br>
- Detaillierte Beschreibungen zum Verfahren finden Sie auch in Küpper, H.-U./Helber, S.: Ablauforganisation in Produktion und Logistik, 3. Aufl., Stuttgart 2004, S. 219f.
</details>

In [103]:
for machine in machineTimes.keys():
    print(machine)

0
1
2


In [112]:
list = []
# check if list is empty
if list:
    print('list is not empty')

In [121]:
def getBottleneck(zeiten_dict):
    final_completion_times = {}
    permutation_machines = {}
    for machine in machineTimes.keys():
        current_time = 0
        jobs_not_scheduled = [i for i in range(len(machineTimes[machine]['B']))]
        permutation_machines[machine] = []
        completion_time = 0

        while jobs_not_scheduled:
            ready_jobs = []
            for job in jobs_not_scheduled:
                if machineTimes[machine]['V'][job] <= current_time:
                    ready_jobs.append(job)
            if ready_jobs:
                # select job with highest Nachlaufzeit
                selected_job = max(ready_jobs, key=lambda x: machineTimes[machine]['N'][x])
                # schedule job, update current time, track completion time, add to permutation
                jobs_not_scheduled.remove(selected_job)
                current_time += machineTimes[machine]['B'][selected_job]
                completion_time = max(completion_time, current_time+machineTimes[machine]['N'][selected_job])
                permutation_machines[machine].append(selected_job)
            else:
                # no job is ready, update current_time to min Vorlaufzeit of unscheduled jobs
                current_time = min(machineTimes[machine]['V'][job] for job in jobs_not_scheduled)
        
        # store the final completion time of the current machine
        final_completion_times[machine] = completion_time
        # print completion time and permutation of the machine
        print(f'Maschine {machine+1}: Permutation {permutation_machines[machine]}, Zykluszeit {completion_time}')

    # return the bottleneck machine with highest completion time and its permutation
    bottleneck_machine = max(final_completion_times, key=final_completion_times.get)
    return f'Engpassmaschine: {bottleneck_machine+1}, Zykluszeit: {final_completion_times[bottleneck_machine]}'

In [122]:
getBottleneck(machineTimes)

Maschine 1: Permutation [0, 2, 1], Zykluszeit 31
Maschine 2: Permutation [2, 1, 0], Zykluszeit 27
Maschine 3: Permutation [1, 0, 2], Zykluszeit 27


'Engpassmaschine: 1, Zykluszeit: 31'